# 04. 워크플로 패턴 — 5가지 핵심 패턴

## 학습 목표

Prompt Chaining, Parallelization, Routing, Orchestrator-Worker, Evaluator-Optimizer 패턴을 이해합니다.

## 4.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 4.2 Prompt Chaining — 순차적 LLM 호출

- 각 단계의 출력이 다음 단계의 입력이 됨
- 용도: 번역 → 검증 → 교정, 분석 → 요약 → 포맷팅

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ChainState(TypedDict):
    topic: str
    draft: str
    improved: str

def generate_draft(state: ChainState) -> dict:
    response = model.invoke(f"다음에 대한 한 문장 사실을 작성해주세요: {state['topic']}", config=lf_config)
    return {"draft": response.content}

def improve_draft(state: ChainState) -> dict:
    response = model.invoke(f"다음 텍스트를 더 매력적으로 개선해주세요: {state['draft']}", config=lf_config)
    return {"improved": response.content}

builder = StateGraph(ChainState)
builder.add_node("draft", generate_draft)
builder.add_node("improve", improve_draft)
builder.add_edge(START, "draft")
builder.add_edge("draft", "improve")
builder.add_edge("improve", END)

chain = builder.compile()
result = chain.invoke({"topic": "Python 프로그래밍"}, config=lf_config)
print(f"초안: {result['draft']}")
print(f"개선: {result['improved']}")

초안: Python은 1991년에 귀도 반 로썸이 발표한 고급 프로그래밍 언어로, 간결하고 읽기 쉬운 문법을 가지고 있습니다.
개선: Python은 1991년, 귀도 반 로썸이 세상에 선보인 고급 프로그래밍 언어로, 간결하고 직관적인 문법 덕분에 누구나 쉽게 배우고 활용할 수 있습니다.


## 4.3 Parallelization — 독립적 태스크의 동시 실행

In [4]:
from typing import Annotated, TypedDict
import operator


class ParallelState(TypedDict):
    text: str
    analyses: Annotated[list[str], operator.add]


def analyze_tone(state: ParallelState) -> dict:
    r = model.invoke(
        f"한 문장으로 다음의 어조를 설명해주세요: {state['text']}",
        config=lf_config
    )
    return {
        "analyses": [f"어조: {r.content}"]
    }


def analyze_complexity(state: ParallelState) -> dict:
    r = model.invoke(
        f"한 문장으로 다음의 복잡도를 평가해주세요: {state['text']}",
        config=lf_config
    )
    return {
        "analyses": [f"복잡도: {r.content}"]
    }


def synthesize(state: ParallelState) -> dict:
    return {
        "analyses": [
            f"종합: {len(state['analyses'])}개 분석 완료"
        ]
    }


builder = StateGraph(ParallelState)

builder.add_node("tone", analyze_tone)
builder.add_node("complexity", analyze_complexity)
builder.add_node("synthesize", synthesize)

# START에서 두 분석 노드로 병렬 분기
builder.add_edge(START, "tone")
builder.add_edge(START, "complexity")

# 두 노드 완료 후 합성
builder.add_edge("tone", "synthesize")
builder.add_edge("complexity", "synthesize")
builder.add_edge("synthesize", END)

parallel_graph = builder.compile()

result = parallel_graph.invoke(
    {
        "text": "LangGraph는 강력한 에이전트 워크플로를 가능하게 합니다.",
        "analyses": []
    },
    config=lf_config
)

for a in result["analyses"]:
    print(f"  {a}")

  복잡도: 주어진 문장의 복잡도는 낮은 편입니다.
  어조: 해당 문장은 LangGraph의 효능을 자신 있게 강조하며 신뢰와 전문성을 전달하는, 단정적이고 확신에 찬 어조입니다.
  종합: 2개 분석 완료


## 4.4 Routing — 분류 기반 분기

![조건부 라우팅 — classify→technical/business/casual](../assets/images/conditional_routing.png)

In [5]:
from pydantic import BaseModel, Field
from typing import Literal, TypedDict


class Classification(BaseModel):
    category: Literal["technical", "business", "casual"]


class RouteState(TypedDict):
    question: str
    category: str
    answer: str


def classify(state: RouteState) -> dict:
    structured = model.with_structured_output(Classification)
    result = structured.invoke(
        f"다음 질문을 분류하세요: {state['question']}",
        config=lf_config
    )
    return {"category": result.category}


def handle_technical(state: RouteState) -> dict:
    r = model.invoke(
        f"기술 전문가로서 답변해주세요: {state['question']}",
        config=lf_config
    )
    return {"answer": r.content}


def handle_business(state: RouteState) -> dict:
    r = model.invoke(
        f"비즈니스 어드바이저로서 답변해주세요: {state['question']}",
        config=lf_config
    )
    return {"answer": r.content}


def handle_casual(state: RouteState) -> dict:
    r = model.invoke(
        f"가볍게 답변해주세요: {state['question']}",
        config=lf_config
    )
    return {"answer": r.content}


def route(state: RouteState) -> str:
    return state["category"]


builder = StateGraph(RouteState)

builder.add_node("classify", classify)
builder.add_node("technical", handle_technical)
builder.add_node("business", handle_business)
builder.add_node("casual", handle_casual)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route)

builder.add_edge("technical", END)
builder.add_edge("business", END)
builder.add_edge("casual", END)

router = builder.compile()

result = router.invoke(
    {
        "question": "Python에서 가비지 컬렉션은 어떻게 작동하나요?"
    },
    config=lf_config
)

print(f"카테고리: {result['category']}")
print(f"답변: {result['answer'][:200]}")

D:\deepagents\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='technical'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(


카테고리: technical
답변: 네, 기술 전문가 관점에서 Python의 가비지 컬렉션(Garbage Collection, GC) 작동 원리를 상세히 설명드리겠습니다.

---

### 1. 기본 원칙

Python의 가비지 컬렉션은 **자동 메모리 관리**를 의미합니다. 프로그래머가 직접 메모리 할당/해제를 하지 않아도, 사용이 끝난 객체(=더 이상 참조되지 않는 객체)를 Python 


## 4.5 Orchestrator-Worker — Send()로 동적 워커 생성

![Orchestrator-Worker 패턴 — plan→Send()→worker(병렬)](../assets/images/orchestrator_worker.png)

In [6]:
from langgraph.types import Send
from typing import Annotated, TypedDict
import operator


class OrchestratorState(TypedDict):
    topic: str
    sections: list[str]
    results: Annotated[list[str], operator.add]


class WorkerState(TypedDict):
    section: str


def plan_sections(state: OrchestratorState) -> dict:
    r = model.invoke(
        f"'{state['topic']}'에 대한 기사의 짧은 섹션 제목 3개를 나열해주세요. 한 줄에 하나씩, 번호 없이.",
        config=lf_config
    )

    sections = [
        s.strip()
        for s in r.content.strip().split("\n")
        if s.strip()
    ][:3]

    return {"sections": sections}


def assign_workers(state: OrchestratorState) -> list[Send]:
    return [
        Send("worker", {"section": s})
        for s in state["sections"]
    ]


def worker(state: WorkerState) -> dict:
    r = model.invoke(
        f"다음에 대해 한 문장으로 작성해주세요: {state['section']}",
        config=lf_config
    )

    return {
        "results": [
            f"## {state['section']}\n{r.content}"
        ]
    }


builder = StateGraph(OrchestratorState)

builder.add_node("plan", plan_sections)
builder.add_node("worker", worker)

builder.add_edge(START, "plan")

builder.add_conditional_edges(
    "plan",
    assign_workers,
    ["worker"]
)

builder.add_edge("worker", END)

orchestrator = builder.compile()

result = orchestrator.invoke(
    {
        "topic": "머신러닝",
        "sections": [],
        "results": []
    },
    config=lf_config
)

for r in result["results"]:
    print(r)
    print()

## 머신러닝이란 무엇인가
머신러닝이란 컴퓨터가 명시적으로 프로그래밍되지 않아도 데이터로부터 학습하여 스스로 성능을 개선하는 인공지능의 한 분야입니다.

## 주요 응용 분야 소개
주요 응용 분야는 해당 기술이나 연구가 실제로 활용되는 대표적인 영역들을 소개하는 것입니다.

## 미래 전망과 도전 과제
미래에는 기술의 발전과 글로벌화로 새로운 기회가 창출되는 한편, 불확실성과 경쟁 심화, 사회적 변화 등이 주요 도전 과제로 부상하고 있습니다.



## 4.6 Evaluator-Optimizer — 생성-평가 반복 루프

In [7]:
class EvalState(TypedDict):
    task: str
    draft: str
    feedback: str
    is_good: bool
    iterations: int

def generate(state: EvalState) -> dict:
    if state.get("feedback"):
        prompt = f"피드백을 반영하여 개선해주세요.\n원본: {state['draft']}\n피드백: {state['feedback']}"
    else:
        prompt = f"다음에 대한 한 문장 슬로건을 작성해주세요: {state['task']}"
    r = model.invoke(prompt, config=lf_config)
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

def evaluate(state: EvalState) -> dict:
    r = model.invoke(f"이 슬로건을 1-10으로 평가하고 간단한 피드백을 주세요: '{state['draft']}'", config=lf_config)
    content = r.content
    is_good = any(f"{n}/10" in content for n in range(8, 11))
    return {"feedback": content, "is_good": is_good}

def should_retry(state: EvalState) -> str:
    if state["is_good"] or state["iterations"] >= 3:
        return END
    return "generate"

builder = StateGraph(EvalState)
builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)

builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")
builder.add_conditional_edges("evaluate", should_retry, ["generate", END])

optimizer = builder.compile()
result = optimizer.invoke({"task": "Python 학습 플랫폼"}, config=lf_config)
print(f"최종 초안 ({result['iterations']}번 반복 후): {result['draft']}")

최종 초안 (2번 반복 후): 피드백을 반영하여 슬로건을 개선해보았습니다. 타깃을 구체화하고, 플랫폼의 차별점을 강조하여 더 명확하고 매력적으로 수정하였습니다.

---

**1. 타깃이 초보자인 경우:**  
쉽고 재미있게, 파이썬 초보의 든든한 시작 플랫폼!

**2. 성장과 실력 향상 강조:**  
실력이 쑥쑥 자라는 쉽고 재미있는 파이썬 학습 플랫폼!

**3. 차별점(맞춤형/커리큘럼 등) 강조:**  
나에게 꼭 맞는 커리큘럼, 쉽고 재미있게 배우는 파이썬!

**4. 전체 추천(종합):**  
쉽고 재미있게, 초보도 금방 실력 UP! 파이썬 성장 플랫폼

---

필요에 따라 타깃층(학생, 실무자, 취업준비생 등)이나 플랫폼의 특장점(예: 실전 예제, 피드백 제공, 커뮤니티 등)을 더 구체적으로 넣어 맞춤형으로 수정할 수도 있습니다! 더 구체적인 타깃이나 강조하고 싶은 플랫폼 특징이 있다면 공유해 주세요.


## 4.7 패턴 비교표

| 패턴 | 결정적 | 병렬 | 반복 | 적합 상황 |
|------|--------|------|------|----------|
| Prompt Chaining | O | X | 순차 | 단계별 변환 |
| Parallelization | O | O | X | 독립 분석 |
| Routing | O | X | X | 분류 기반 처리 |
| Orchestrator-Worker | O | O | X | 동적 하위 작업 |
| Evaluator-Optimizer | X | X | O | 품질 개선 루프 |

### 다음 단계
→ **[05_agents.ipynb](./05_agents.ipynb)**: ReAct 에이전트를 구축합니다.
